<a href="https://colab.research.google.com/github/Shirley31415926/Greening_Juneau_Icefield/blob/main/access_ndvi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install earthengine-api pandas


In [ ]:
import ee
import sys
import datetime
import pandas as pd
import traceback


In [ ]:
ee.Authenticate()
#Please replace with 'your-earthengine-project-id'
ee.Initialize(project='ee-huang31415926')
print("EE initialized")

EE initialized


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# small function for adding 'site' property to randomly created points
def addID(pt):
    return pt.set('site', pt.id())


# SPECIFY SEVERAL VARIABLES
BUFFER_DIST = 0  # set it to 0 if buffer is not needed, else it will make script timed-out
CRS = 'EPSG:4326'
SCALE = 30
NULL_VALUE = 0
MASK_VALUE = 0  # used for ADDON image
# startJulian = 91  # early April
# endJulian = 150  # late May
startJulian = 150  # late may
endJulian = 250  # early sept


# values to copy from fusion table/feature collection
feat_properties = ['site', 'CID', 'ORIG_FID']
KEYS = ee.List(feat_properties)

# CHANGED: update/trim metadata keys for Landsat C2 L2
scene_properties = [
    'CLOUD_COVER',
    'LANDSAT_PRODUCT_ID',
    'SUN_ELEVATION',
    'SUN_AZIMUTH'
]

PROPERTY_LIST = ee.List(scene_properties)

# Bands to retrieve
# CHANGED: remove SR_B6 (doesn't exist in LT05/LE07 C02 L2 SR bands)
bands = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7','QA_PIXEL','QA_RADSAT']

BAND_LIST = ee.List(bands)

# addon asset and bands
ADDON = ee.Image('JRC/GSW1_0/GlobalSurfaceWater').float().unmask(MASK_VALUE)
ADDON_BANDLIST = ee.List(['max_extent'])

import os

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
OUTFILE = f'/content/drive/MyDrive/juneau/lsat_tundra_samples_{timestamp}.csv'

# ensure output directory exists
out_dir = os.path.dirname(OUTFILE)
os.makedirs(out_dir, exist_ok=True)

# Landsat Surface Reflectance collections
ls5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")   # CHANGED
ls7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")   # CHANGED
ls8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")   # CHANGED

# define region of interest
sites = ee.FeatureCollection('projects/ee-huang31415926/assets/juneau_foreland_sites_buf10km_noWater_1000')
sites = sites.map(addID)
boundary = sites.geometry().bounds()

/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for JRC/GSW1_0/GlobalSurfaceWater! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_0_GlobalSurfaceWater

  warnings.warn(warning, category=DeprecationWarning)


In [ ]:
# ------------------------------------------------------------------------------------
# BE CAREFUL MODIFYING THE STUFF BELOW
# ------------------------------------------------------------------------------------

def add_lists(list1, list2):
    first = ee.List(list1).add(list2.get(0))
    n = list2.size()

    def _combine(this, prev):
        return ee.List(prev).add(this)

    return ee.List(list2).slice(1, n).iterate(_combine, first)


ALL_BANDS = add_lists(BAND_LIST, ADDON_BANDLIST)

# CHANGED: merge collections
LS_COLL = (ee.ImageCollection(ls5.merge(ls7).merge(ls8))
           .filter(ee.Filter.calendarRange(startJulian, endJulian, "day_of_year"))
           .filterBounds(boundary)
           .map(lambda image: image.addBands(ADDON, ADDON_BANDLIST))
           .select(ALL_BANDS)
           .map(lambda image: image.float())
          )


def cprint(text):
    sys.stdout.write(str(text) + '\n')
    sys.stdout.flush()


def eprint(excep):
    print(excep)
    print('---------------------------------------------------------------------')
    print('Error Traceback:')
    print(traceback.format_exc())
    print('---------------------------------------------------------------------')


# function to extract properties from collection as list of dictionaries
def get_coll_dict(image_collection):
    n_images = ee.ImageCollection(image_collection).size()
    coll_list = ee.ImageCollection(image_collection).toList(n_images)

    def get_prop(image):
        image_dict = ee.Image(image).toDictionary()
        val_list = (PROPERTY_LIST
            .map(lambda prop: image_dict.get(prop))
            .add(ee.Image(image).get('system:index')))
        return ee.Dictionary.fromLists(PROPERTY_LIST.add('system:index'), val_list)

    return coll_list.map(get_prop)


# function to extract a feature from a non-null collection
def get_region(feature):

    coll = LS_COLL.filterBounds(ee.Feature(feature).geometry())
    ncoll = coll.size().getInfo()

    # feature to be extracted
    feat = ee.Algorithms.If(ee.Number(BUFFER_DIST).gt(0),
                            ee.Feature(feature).buffer(BUFFER_DIST).bounds(),
                            ee.Feature(feature))

    feat_dict = feat.getInfo()
    feat_props = feat_dict['properties']

    pt_list = list()

    if ncoll > 0:

        coll_dicts = get_coll_dict(coll).getInfo()

        temp_list = coll.getRegion(ee.Feature(feat).geometry(), SCALE).getInfo()
        n = len(temp_list)

        elem_names = temp_list[0]
        id_column = elem_names.index('id')

        for j in range(1, n):
            pt_dict = dict()
            for k in range(0, len(elem_names)):
                pt_dict[elem_names[k]] = temp_list[j][k]
                id = temp_list[j][id_column]

                scene_prop_list = [d for d in coll_dicts if d['system:index'] == id]
                if not scene_prop_list:
                  continue
                scene_prop = scene_prop_list[0]

                for prop in scene_properties:
                    pt_dict[prop] = scene_prop[prop]

                for prop in feat_properties:
                    pt_dict[prop] = feat_props[prop]

            pt_list.append(pt_dict)

    else:
        pt_dict = dict()
        pt_dict['id'] = NULL_VALUE
        pt_dict['time'] = NULL_VALUE

        for prop in feat_properties:
            pt_dict[prop] = feat_props[prop]

        for prop in scene_properties:
            pt_dict[prop] = NULL_VALUE

        for band in bands:
            pt_dict[band] = NULL_VALUE

        pt_list.append(pt_dict)

    return pt_list


time0 = datetime.datetime.now()

n_sites = sites.size().getInfo()
sites_list = sites.toList(n_sites)

for i in range(0, n_sites):
    time1 = datetime.datetime.now()
    site = sites_list.get(i)

    try:
        temp_dicts = get_region(site)

    # if any error occurs, print it and break the loop
    except Exception as e:
        eprint(e)
        break

    df = pd.DataFrame(temp_dicts)

    if i == 0:
        with open(OUTFILE, 'w') as f:
            df.to_csv(f, index=False, header=True)
    else:
        with open(OUTFILE, 'a') as f:
            df.to_csv(f, index=False, header=False)

    time2 = datetime.datetime.now()

    cprint('Time taken for sample {s}: {t} seconds'.format(s=str(i + 1),
                                                           t=str(round((time2 - time1).total_seconds(), 1))))

time0_ = datetime.datetime.now()
cprint('Total Time taken: {t} seconds'.format(t=str(round((time0_ - time0).total_seconds(), 1))))

Time taken for sample 1: 12.6 seconds
Time taken for sample 2: 14.9 seconds
Time taken for sample 3: 12.9 seconds
Time taken for sample 4: 12.5 seconds
Time taken for sample 5: 18.4 seconds
Time taken for sample 6: 14.4 seconds
Time taken for sample 7: 14.4 seconds
Time taken for sample 8: 36.3 seconds
Time taken for sample 9: 11.7 seconds
Time taken for sample 10: 23.3 seconds
Time taken for sample 11: 19.7 seconds
Time taken for sample 12: 12.0 seconds
Time taken for sample 13: 15.8 seconds
Time taken for sample 14: 13.3 seconds
Time taken for sample 15: 12.3 seconds
Time taken for sample 16: 19.3 seconds
Time taken for sample 17: 19.2 seconds
Time taken for sample 18: 15.8 seconds
Time taken for sample 19: 15.2 seconds
Time taken for sample 20: 10.9 seconds
Time taken for sample 21: 13.1 seconds
Time taken for sample 22: 14.2 seconds
Time taken for sample 23: 39.8 seconds
Time taken for sample 24: 15.7 seconds
Time taken for sample 25: 16.0 seconds
Time taken for sample 26: 16.7 sec

Time taken for sample 214: 17.6 seconds
Time taken for sample 215: 20.7 seconds
Time taken for sample 216: 14.5 seconds
Time taken for sample 217: 16.9 seconds
Time taken for sample 218: 10.5 seconds
Time taken for sample 219: 15.3 seconds
Time taken for sample 220: 11.4 seconds
Time taken for sample 221: 13.8 seconds
Time taken for sample 222: 17.0 seconds
Time taken for sample 223: 14.8 seconds
Time taken for sample 224: 18.8 seconds


Time taken for sample 225: 13.3 seconds
Time taken for sample 226: 16.6 seconds
Time taken for sample 227: 13.4 seconds
Time taken for sample 228: 19.3 seconds
Time taken for sample 229: 14.8 seconds
Time taken for sample 230: 11.9 seconds


Time taken for sample 231: 23.5 seconds
Time taken for sample 232: 22.6 seconds
Time taken for sample 233: 11.8 seconds
Time taken for sample 234: 17.1 seconds
Time taken for sample 235: 15.1 seconds


Time taken for sample 236: 23.6 seconds
Time taken for sample 237: 16.4 seconds
Time taken for sample 238: 11.4 seconds
Time taken for sample 239: 13.3 seconds
Time taken for sample 240: 13.4 seconds
Time taken for sample 241: 14.2 seconds
Time taken for sample 242: 10.3 seconds
Time taken for sample 243: 17.6 seconds
Time taken for sample 244: 19.5 seconds
Time taken for sample 245: 14.1 seconds
Time taken for sample 246: 12.4 seconds
Time taken for sample 247: 13.8 seconds
Time taken for sample 248: 10.8 seconds
Time taken for sample 249: 15.5 seconds
Time taken for sample 250: 17.4 seconds
Time taken for sample 251: 16.0 seconds
Time taken for sample 252: 15.4 seconds
Time taken for sample 253: 12.4 seconds
Time taken for sample 254: 12.8 seconds
Time taken for sample 255: 12.9 seconds
Time taken for sample 256: 14.3 seconds
Time taken for sample 257: 24.9 seconds
Time taken for sample 258: 14.0 seconds
Time taken for sample 259: 13.4 seconds
Time taken for sample 260: 11.4 seconds
